In [24]:
#Data collection
import nltk
nltk.download("gutenberg")
from nltk.corpus import gutenberg
import pandas as pd

##load the dataset
data = gutenberg.raw("shakespeare-hamlet.txt")
with open("hamlet.txt", "w") as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to C:\Users\Akshay R
[nltk_data]     K\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [25]:
##Data preprocessing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

##load dataset
with open("hamlet.txt", "r") as file:
    text = file.read().lower()

##Tokenize the text - creating indexes for words
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index)+1

In [26]:
##Create input sequences
input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [27]:
##Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [28]:
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre"))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

In [29]:
##create predictors and label
import tensorflow as tf
x, y = input_sequences[:, :-1], input_sequences[:, -1]
y

array([ 687,    4,   45, ..., 1047,    4,  193],
      shape=(25732,), dtype=int32)

In [30]:
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
y


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

In [31]:
#Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [ ]:
##Train LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GRU

##Define the model
model = Sequential()
model.add(Embedding(total_words, 100))
model.build(input_shape=(None, max_sequence_len))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation="softmax"))

##Compile the model
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 14, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 14, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
##Train with GRU LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GRU

##Define the model
model = Sequential()
model.add(Embedding(total_words, 100))
model.build(input_shape=(None, max_sequence_len))
model.add(GRU(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(100))
model.add(Dense(total_words, activation="softmax"))

##Compile the model
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

In [34]:
##Train the model
history = model.fit(x_train, y_train, epochs=50, validation_data=(x_test, y_test), verbose=1)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 35s 48ms/step - accuracy: 0.0334 - loss: 6.9000 - val_accuracy: 0.0340 - val_loss: 6.7443
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 32s 49ms/step - accuracy: 0.0399 - loss: 6.4494 - val_accuracy: 0.0420 - val_loss: 6.8166
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.0458 - loss: 6.2994 - val_accuracy: 0.0507 - val_loss: 6.8597
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 32s 49ms/step - accuracy: 0.0521 - loss: 6.1649 - val_accuracy: 0.0513 - val_loss: 6.8839
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 30s 46ms/step - accuracy: 0.0556 - loss: 6.0365 - val_accuracy: 0.0575 - val_loss: 6.9125
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.0629 - loss: 5.9029 - val_accuracy: 0.0620 - val_loss: 6.9681
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 37s 58ms/step - accuracy: 0.0684 - loss: 5.7653 - val_accuracy: 0.0657 - val_loss: 7.0222
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 45s 70ms/step - accuracy: 0.0772 - loss: 5.6275 - 

In [35]:
# Function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]  # Ensure the sequence length matches max_sequence_len-1
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [36]:
input_text="To be or not to be"
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:To be or not to be
Next Word PRediction:blest


In [37]:
## Save the model
model.save("next_word_lstm.h5")
## Save the tokenizer
import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)

In [38]:
input_text="  Barn. Last night of all,When yond same"
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:  Barn. Last night of all,When yond same
Next Word PRediction:vngracious
